# Mud-chip (rip-up clast) grain size from µXRF Tornado element maps
Measures mud chips **directly on the M4 Tornado map**, because H₂O₂ destroyed them before the Malvern run.

**How it works (one line per step)**
1. Load the element map (colour TIFF or raw counts) and get the pixel size (Tornado step size, or the burned-in 3 cm scale bar).
2. Build a core mask and blank out cracks, holes and the text/scale bar.
3. Divide out slow down-core brightness drift, threshold, clean speckle, split touching chips (watershed).
4. Keep only objects ≥ 3 px across **and** clearly brighter than the ring of matrix around them.
5. Measure every chip (ESD, Feret min/max, ellipse axes, shape) and drop chips cut by the core edge, re-weighting the rest (Miles–Lantuéjoul).
6. Compute D10/D50/D90 and Folk & Ward stats (volume-weighted so they sit alongside the Malvern numbers), a down-core profile, and a sensitivity test.

Run the cells in order. **Only Cell 2 needs editing.**

### Cell 1 — Setup
Installs `tifffile` + `imagecodecs` (needed for LZW/Deflate/JPEG-compressed TIFFs, which is how most software saves them), imports libraries, and opens the upload dialog. If a TIFF still won't open, the loader automatically falls back to Pillow and then OpenCV, and tells you exactly why if all three fail. Upload the element-map TIFF(s) exported from the M4 Tornado software **at original resolution** (a screenshot is much coarser — every pixel becomes 5–6× bigger).

In [ ]:
# =====================================================================
# CELL 1 — Setup: install, import, upload the Tornado element map(s)
# =====================================================================
%pip install -q tifffile imagecodecs scikit-image   # imagecodecs = LZW/Deflate/JPEG-compressed TIFFs

import os, glob, math, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
from PIL import Image
from scipy import ndimage as ndi
from scipy.spatial import ConvexHull
from skimage import filters, measure, morphology, segmentation
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

UPLOAD_NOW = True   # set False if the files are already in /content (or use Drive)

if IN_COLAB and UPLOAD_NOW:
    uploaded = files.upload()            # pick your .tif/.tiff element map(s)
    print("Uploaded:", list(uploaded.keys()))

img_files = sorted(f for f in os.listdir(".")
                   if f.lower().endswith((".tif", ".tiff", ".png", ".jpg", ".jpeg")))
print("Image files here:", img_files)


### Cell 2 — Config (edit this)
- **Depth:** the scan is just a strip of pixels and does not know where it sits in the core. `CORE_TOP_SIDE` says which end of the image is the shallower one; `SCAN_TOP_DEPTH_CM` and `SCAN_BOTTOM_DEPTH_CM` are the core depths of the two ends. Give both if you know them - depth then comes from your core log and Cell 4 cross-checks it against the pixel size.
- **Events:** point `EVENT_FILE` at your event list and everything is reported per event, in `BIN_MM` (2 mm) bins.
- `SEG_FILE`: the map where chips stand out best from the matrix. Look at a few elements and pick the one where you'd trace the chips by eye most easily. If chips are darker than the matrix, set `INVERT = True`.
- `PIXEL_SIZE_UM`: the mosaic step size from the Tornado. Leave `None` to calibrate from the scale bar.
- `INTERVALS`: depth ranges (cm) to measure. Run Cell 4 once to read depths off the ruler, then come back and set them.
- Segmentation knobs: start with the defaults, check the QC figure in Cell 5, adjust only if chips are clearly missed or matrix is picked up.

In [ ]:
# =====================================================================
# CELL 2 — CONFIG (the only cell you normally edit)
# =====================================================================
CORE_ID = "LACHUA-24A-1G-1"   # <- your lab's core/section label for Lake Lachua core 24A

# --- Input maps --------------------------------------------------------
# Map used to find the mud chips (the element where chips contrast best
# with the matrix). Export from M4 Tornado at ORIGINAL resolution
# (not a screenshot) so 1 pixel = 1 step.
SEG_FILE = "Lachua_24A_Al.tif"    # Aluminum (Al) map from the M4 Tornado - rename to your exported file.
                                   # run Cell 3b to confirm Al is the best-contrast element for this core
SEG_PAGE = 0        # if the TIFF holds several element maps, which page (see list printed in Cell 4)

# Optional: other element maps of the SAME scan (same size). The mean
# intensity of each map inside every chip is added to the results table.
EXTRA_MAPS = {
    # "Ca": "Video 1_Ca.tif",          # separate file per element, or
    # "Fe": ("tornado_all.tif", 3),    # (file, page) for a multi-page TIFF
}

INVERT = False      # True if chips are DARKER than the matrix in SEG_FILE

# --- Pixel size ----------------------------------------------------------
# Best: the step size (µm) of the Tornado mosaic, read from the M4 software.
# If None, the code measures the white scale bar burned into the image.
PIXEL_SIZE_UM = None
SCALEBAR_UM   = 30000          # length the scale bar represents (3 cm = 30000 µm)
SCALEBAR_FILE = None           # file carrying the burned-in scale bar, e.g. "Video Mosaic.tif".
                               # None = use SEG_FILE. Must have the same pixel dimensions.

# --- Depth registration -------------------------------------------------
# The scan is a strip of pixels; it has no idea where it sits in the core.
# Tell it the core depth at each end and everything downstream is in cm.
CORE_TOP_SIDE        = "left"   # which END of the image is the shallower (up-core) end
SCAN_TOP_DEPTH_CM    = 0.0      # core depth at that end
SCAN_BOTTOM_DEPTH_CM = None     # core depth at the other end. Give BOTH ends if you know
                                # them (best: depth then comes from your core log, and the
                                # code cross-checks it against the pixel size). Leave None
                                # to derive depth from the pixel size instead.

# --- Optional: find the scan depth automatically (Cell 4b) --------------
# If you do not know where the scan sits in the core, give your Avaatech
# XRF spreadsheet (depth + counts). Cell 4b slides the Tornado profile
# along it and reports the depth that matches best.
XRF_FILE     = None            # e.g. "LACHUA-24A-1G-1_Repo_RAW_XRF.xlsx"; None = skip
XRF_SHEET    = 0
XRF_DEPTH_COL = None           # None = auto-detect a "depth" column
XRF_ELEMENT  = "Ca"            # element column in that file to match on
XRF_UNITS    = "cm"            # units of its depth column ("cm" or "mm")
MATCH_FILE   = None            # Tornado map of the SAME element, e.g. "Video 1_Ca.tif"
                               # None = use SEG_FILE

# --- Events (turbidites) ------------------------------------------------
EVENT_FILE  = "Lachua_24A_events.csv"  # CSV/XLSX, one row per event: name, top, bottom
                                # (+ optional type column). None = use INTERVALS below.
EVENT_SHEET = 0                 # sheet name/number if EVENT_FILE is an Excel file
EVENT_UNITS = "cm"              # "cm" or "mm" - units of the depth columns in that file
EVENT_PAD_CM = 0.0              # extra depth added above and below each event before measuring
BIN_MM = 2.0                    # down-core bin size inside each event

# --- Intervals: used only when EVENT_FILE is None ----------------------
INTERVALS = [
    # (name,               top_cm, bottom_cm)
    ("chip_zone_1",          13.4,   16.8),   # only used if EVENT_FILE is None/missing
]

# --- Segmentation ---------------------------------------------------------
SMOOTH_SIGMA_PX   = 0.7     # light denoising (pixel noise); 0 = off
FLATTEN_SIGMA_MM  = 2.0     # removes slow down-core brightness drift; 0 = off
THRESH_METHOD     = "otsu"  # "otsu", "li", "triangle", or "manual"
THRESH_MANUAL     = 1.10    # only used if THRESH_METHOD == "manual"
MIN_DIAM_PX       = 3       # resolution limit: chips < 3 px across are discarded
CONTRAST_MIN      = 1.20    # chip mean / surrounding-ring mean must exceed this
SPLIT_TOUCHING    = True    # watershed split of touching chips
SPLIT_H_PX        = 1.0     # neck depth (px) needed before two touching chips are split
DROP_EDGE_CHIPS   = True    # drop chips cut by the interval/core edge and re-weight
                            # the rest (Miles–Lantuéjoul correction for edge bias)

# --- Size statistics ----------------------------------------------------
SIZE_METRIC   = "esd_um"    # "esd_um", "feret_min_um", "feret_max_um", "minor_axis_um"
WEIGHTING     = "volume"    # "volume" (comparable to Malvern vol.%) or "number"
APPLY_STEREO  = False       # multiply 2-D section sizes by 4/π (sphere correction)

OUT_DIR = "mudchip_results"
os.makedirs(OUT_DIR, exist_ok=True)

# --- sanity check: are the files you named actually here? ---------------
_here = sorted(f for f in os.listdir(".")
               if f.lower().endswith((".tif", ".tiff", ".png", ".csv", ".xlsx", ".xls")))
for _var, _f in [("SEG_FILE", SEG_FILE), ("SCALEBAR_FILE", SCALEBAR_FILE),
                 ("EVENT_FILE", EVENT_FILE), ("XRF_FILE", XRF_FILE), ("MATCH_FILE", MATCH_FILE)]:
    if _f and not os.path.exists(_f if isinstance(_f, str) else _f[0]):
        raise FileNotFoundError(
            f"{_var} = '{_f}' is not in this folder. Edit the {_var} line above to one of "
            "these names (copy it exactly, spaces included), then run this cell again.\n"
            "If you have not uploaded it yet, re-run Cell 1 and upload it, or set the option "
            f"to None to skip it.\nFiles in this folder:\n  " + "\n  ".join(_here))
print(f"Config loaded for {CORE_ID} | SEG_FILE = {SEG_FILE}"
      + (f" | scale bar from {SCALEBAR_FILE}" if SCALEBAR_FILE else ""))


### Cell 3 — Helper functions
All the machinery lives here so the later cells stay short. Key ideas:
- **Colour maps:** element signal = colour strength (brightest channel − darkest channel), which also removes grey/white labels automatically.
- **Flattening:** each pixel is divided by the local average of the matrix (Gaussian, `FLATTEN_SIGMA_MM`), so a chip is judged against its own neighbourhood, not the whole core.
- **Contrast test:** each candidate's mean brightness is compared to a 2-px ring around it; noise speckles fail, real chips pass.
- **Feret min/max:** measured on the convex hull of the pixel corners (calliper widths), so tiny chips are not underestimated by one pixel.
- **Edge correction:** chips touching the core edge or interval ends are removed, and every remaining chip gets weight W·H / ((W−w)(H−h)) so large chips (more likely to be cut) are not under-counted.
- **Folk & Ward:** φ-percentiles are read from the coarse-to-fine cumulative curve, same formulas as your Malvern stats.

In [ ]:
# =====================================================================
# CELL 3 — Helper functions (run once; no edits needed)
# =====================================================================

# ---------- loading --------------------------------------------------
def _tiff_page(path, page):
    """Read one page of a TIFF with tifffile (handles 16-bit, float, multi-page, palette)."""
    with tifffile.TiffFile(path) as tif:
        n = len(tif.pages)
        if page >= n:
            raise ValueError(f"{path} has {n} page(s); page {page} does not exist")
        pg = tif.pages[page]
        arr = pg.asarray()
        if pg.photometric == tifffile.PHOTOMETRIC.PALETTE and pg.colormap is not None:
            cmap = (np.asarray(pg.colormap) / 257).astype(np.uint8)       # (3, 2**bits)
            arr = np.moveaxis(cmap[:, arr], 0, -1)                          # indices -> RGB
        elif pg.planarconfig == tifffile.PLANARCONFIG.SEPARATE and arr.ndim == 3:
            arr = np.moveaxis(arr, 0, -1)                                   # (C,H,W) -> (H,W,C)
    return arr


def read_image(path, page=0):
    """Read TIFF/PNG/JPG robustly: tifffile -> Pillow -> OpenCV. Returns a numpy array."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"'{path}' not found. Files here: {sorted(os.listdir('.'))}")
    errors = []
    if path.lower().endswith((".tif", ".tiff")):
        try:
            return _tiff_page(path, page)
        except Exception as e:
            errors.append(f"tifffile: {e}")
    try:                                                     # Pillow reads LZW/PackBits natively
        with Image.open(path) as im:
            im.seek(page)
            if im.mode in ("P", "PA", "CMYK", "YCbCr", "LA"):
                im = im.convert("RGB")
            return np.array(im)
    except Exception as e:
        errors.append(f"Pillow: {e}")
    try:
        import cv2
        ok, pages = cv2.imreadmulti(path, flags=cv2.IMREAD_UNCHANGED)
        if ok and len(pages) > page:
            arr = pages[page]
            return arr[..., ::-1] if arr.ndim == 3 else arr   # BGR -> RGB
        errors.append("OpenCV: could not decode")
    except Exception as e:
        errors.append(f"OpenCV: {e}")
    with open(path, "rb") as f:
        head = f.read(8)
    raise ValueError(f"Could not read {path} (first bytes {head!r}).\n  " + "\n  ".join(errors) +
                     "\nA real TIFF starts with b'II*\\x00' or b'MM\\x00*'. If it doesn't, the file is "
                     "not a TIFF (e.g. a renamed .bcf) - re-export it from the M4 software as TIFF or PNG.")


def list_pages(path):
    """Print every page in a TIFF (Tornado can put several element maps in one file)."""
    if not path.lower().endswith((".tif", ".tiff")):
        print(path, ": single image"); return
    try:
        with tifffile.TiffFile(path) as tif:
            for i, pg in enumerate(tif.pages):
                desc = (pg.description or "")[:60].replace("\n", " ")
                print(f"  page {i}: {pg.shape} {pg.dtype} {pg.photometric.name} "
                      f"compression={pg.compression.name} {desc}")
    except Exception as e:
        print("  could not list pages:", e)


def load_map(path, invert=False, page=0):
    """Return (intensity 2-D float, annotation mask or None, rgb or None)."""
    if isinstance(path, (tuple, list)):                      # ("file.tif", page)
        path, page = path
    arr = np.squeeze(read_image(path, page))
    if arr.ndim == 3 and arr.shape[0] in (3, 4) and arr.shape[-1] not in (3, 4):
        arr = np.moveaxis(arr, 0, -1)                        # (C,H,W) -> (H,W,C)
    if arr.ndim == 3 and arr.shape[-1] not in (3, 4):
        print(f"  {path}: stack {arr.shape}, using layer 0")
        arr = arr[0]
    if arr.ndim == 3 and arr.shape[-1] == 3 and np.allclose(arr[..., 0], arr[..., 1]) \
            and np.allclose(arr[..., 1], arr[..., 2]):
        arr = arr[..., 0]                                    # grey saved as RGB -> single channel

    if arr.ndim == 3:                                        # colour-coded map (RGB/RGBA)
        rgb = arr[..., :3].astype(float)
        vmax = np.iinfo(arr.dtype).max if np.issubdtype(arr.dtype, np.integer) else rgb.max()
        hi, lo = rgb.max(axis=2), rgb.min(axis=2)
        intensity = hi - lo                                  # colour strength = element signal
        annot = (lo > 0.35 * vmax) & ((hi - lo) < 0.15 * vmax)   # white/grey text & scale bar
    else:                                                    # raw single-channel counts
        rgb, annot = None, None
        intensity = np.nan_to_num(arr.astype(float))

    if annot is not None:
        intensity[ndi.binary_dilation(annot, iterations=2)] = 0
    if invert:
        intensity = np.nanmax(intensity) - intensity
    return intensity, annot, rgb


def scalebar_length_px(annot):
    """Longest horizontal run of annotation pixels = scale bar length (px)."""
    if annot is None or not annot.any():
        return None
    best = (0, None, None, None)
    for r in np.where(annot.sum(axis=1) > 20)[0]:
        row = np.concatenate([[0], annot[r].astype(int), [0]])
        d = np.diff(row)
        starts, ends = np.where(d == 1)[0], np.where(d == -1)[0]
        lengths = ends - starts
        i = lengths.argmax()
        if lengths[i] > best[0]:
            best = (lengths[i], r, starts[i], ends[i])
    length, r, c0, c1 = best
    if length < 30:
        return None
    return dict(length_px=length - 1, row=r, c0=c0, c1=c1)   # -1 ≈ tick centre to tick centre


def core_mask(intensity, erode_px=3):
    """Core = large bright object; returns (core incl. voids, valid sediment pixels)."""
    thr = 0.15 * np.percentile(intensity[intensity > 0], 99)
    fg = intensity > thr
    core = ndi.binary_closing(fg, structure=morphology.disk(5))
    core = ndi.binary_fill_holes(core)
    lab = measure.label(core)
    sizes = np.bincount(lab.ravel()); sizes[0] = 0
    core = np.isin(lab, np.where(sizes > 0.02 * core.size)[0])
    core = ndi.binary_erosion(core, structure=morphology.disk(erode_px))
    dark = filters.gaussian(intensity, 1, preserve_range=True) < 0.06 * np.percentile(intensity[intensity > 0], 99)
    voids = core & dark                                  # cracks, holes, gas voids (near-zero signal)
    lab_v = measure.label(voids)
    vs = np.bincount(lab_v.ravel()); vs[0] = 0
    voids = np.isin(lab_v, np.where(vs >= 25)[0])        # ignore dark specks < 25 px
    valid = core & ~voids
    return core, valid


# ---------- depth <-> pixel ------------------------------------------
# Distance from the up-core end of the scan, in pixels, for a column index.
def _dist_px(col, width):
    return np.asarray(col, float) if CORE_TOP_SIDE == "left" else (width - 1 - np.asarray(col, float))

def col_to_depth_cm(col, width, px_um):
    """Column index -> core depth in cm."""
    if SCAN_BOTTOM_DEPTH_CM is not None:                       # two-point registration
        span = SCAN_BOTTOM_DEPTH_CM - SCAN_TOP_DEPTH_CM
        return SCAN_TOP_DEPTH_CM + _dist_px(col, width) / (width - 1) * span
    return SCAN_TOP_DEPTH_CM + _dist_px(col, width) * px_um / 1e4

def depth_to_col(depth_cm, width, px_um):
    """Core depth in cm -> column index (clipped to the image)."""
    if SCAN_BOTTOM_DEPTH_CM is not None:
        span = SCAN_BOTTOM_DEPTH_CM - SCAN_TOP_DEPTH_CM
        dist = (depth_cm - SCAN_TOP_DEPTH_CM) / span * (width - 1)
    else:
        dist = (depth_cm - SCAN_TOP_DEPTH_CM) * 1e4 / px_um
    col = dist if CORE_TOP_SIDE == "left" else (width - 1 - dist)
    return int(np.clip(round(float(col)), 0, width))

def scan_depth_range(width, px_um):
    d = [col_to_depth_cm(0, width, px_um), col_to_depth_cm(width - 1, width, px_um)]
    return min(d), max(d)


# ---------- segmentation ---------------------------------------------
def default_params():
    return dict(smooth=SMOOTH_SIGMA_PX, flatten_mm=FLATTEN_SIGMA_MM,
                method=THRESH_METHOD, manual=THRESH_MANUAL, thresh_scale=1.0,
                min_diam_px=MIN_DIAM_PX, contrast=CONTRAST_MIN,
                split=SPLIT_TOUCHING, split_h=SPLIT_H_PX, drop_edge=DROP_EDGE_CHIPS)


def segment_roi(roi, valid, px_um, p):
    """Label mud chips inside one interval. Returns label image, flattened image, threshold."""
    img = filters.gaussian(roi, p["smooth"], preserve_range=True) if p["smooth"] > 0 else roi.copy()
    img[~valid] = 0

    # 1) flatten slow brightness drift: divide by local mean of valid pixels
    if p["flatten_mm"] and p["flatten_mm"] > 0:
        s = p["flatten_mm"] * 1000 / px_um
        w = filters.gaussian(valid.astype(float), s)
        m = filters.gaussian(np.where(valid, img, 0), s) / np.maximum(w, 1e-6)
        flat = np.where(valid, img / np.maximum(m, 1e-6), 0)
    else:
        flat = np.where(valid, img / np.mean(img[valid]), 0)

    # 2) threshold
    vals = flat[valid]
    t = {"otsu": filters.threshold_otsu, "li": filters.threshold_li,
         "triangle": filters.threshold_triangle}.get(p["method"], None)
    thr = (t(vals) if t else p["manual"]) * p["thresh_scale"]
    bw = (flat > thr) & valid

    # 3) clean-up: remove speckle, fill pinholes, drop sub-resolution objects
    bw = ndi.binary_opening(bw, structure=morphology.disk(1))
    bw = ndi.binary_fill_holes(bw)
    min_area = math.pi * (p["min_diam_px"] / 2) ** 2
    lab0 = measure.label(bw)
    sz = np.bincount(lab0.ravel()); sz[0] = 0
    bw = np.isin(lab0, np.where(sz >= min_area)[0])

    # 4) split touching chips (distance-transform watershed with h-maxima markers)
    if p["split"]:
        dist = filters.gaussian(ndi.distance_transform_edt(bw), 0.5)
        markers = measure.label(morphology.h_maxima(np.round(dist * 10).astype(np.int32),
                                                    int(p["split_h"] * 10)) & bw)
        lab = segmentation.watershed(-dist, markers, mask=bw)
        missing = measure.label(bw & (lab == 0))
        lab[missing > 0] = missing[missing > 0] + lab.max()
    else:
        lab = measure.label(bw)
    sz = np.bincount(lab.ravel()); sz[0] = 0                # split pieces must also pass the size limit
    lab = np.where(np.isin(lab, np.where(sz >= min_area)[0]), lab, 0)
    lab = segmentation.relabel_sequential(lab)[0]

    # 5) contrast test: chip must be brighter than a 2-px ring around it
    if lab.max() > 0:
        idx = np.arange(1, lab.max() + 1)
        inner = ndi.mean(flat, lab, idx)
        ring = segmentation.expand_labels(lab, 2)
        ring[(lab > 0) | ~valid] = 0
        outer = np.array(ndi.mean(flat, ring, idx))
        outer = np.where(np.isfinite(outer) & (outer > 0), outer, np.inf)
        keep = idx[(np.array(inner) / outer) >= p["contrast"]]
        lab = np.where(np.isin(lab, keep), lab, 0)
        lab = segmentation.relabel_sequential(lab)[0]
    return lab, flat, thr


# ---------- measurement ----------------------------------------------
_CORNERS = np.array([[-.5, -.5], [-.5, .5], [.5, -.5], [.5, .5]])

def feret_min_max(coords):
    pts = np.unique((coords[:, None, :] + _CORNERS[None]).reshape(-1, 2), axis=0)
    try:
        hp = pts[ConvexHull(pts).vertices]
    except Exception:
        ext = np.ptp(pts, axis=0)
        return float(ext.min()), float(np.hypot(*ext))
    d = hp[:, None, :] - hp[None, :, :]
    fmax = np.sqrt((d ** 2).sum(-1)).max()
    edges = np.roll(hp, -1, axis=0) - hp
    normals = np.stack([-edges[:, 1], edges[:, 0]], 1)
    normals /= np.linalg.norm(normals, axis=1, keepdims=True)
    fmin = np.ptp(hp @ normals.T, axis=0).min()
    return float(fmin), float(fmax)


def measure_chips(lab, roi_core, x0, full_width, px_um, extra_rois=None, drop_edge=True):
    H, W = lab.shape
    # outer rim of the core inside this interval (core liner edges + interval ends)
    edge_zone = roi_core & ~ndi.binary_erosion(roi_core, border_value=0)
    core_h = np.median(roi_core.sum(axis=0))

    rows = []
    for r in measure.regionprops(lab):
        fmin, fmax = feret_min_max(r.coords)
        touches = bool(edge_zone[r.coords[:, 0], r.coords[:, 1]].any())
        bh, bw_ = r.bbox[2] - r.bbox[0], r.bbox[3] - r.bbox[1]
        denom = (W - bw_) * (core_h - bh)
        edge_w = (W * core_h / denom) if denom > 0 else np.nan     # Miles–Lantuéjoul
        perim = r.perimeter_crofton if r.area > 4 else r.perimeter
        col = r.centroid[1] + x0
        row = dict(
            chip_id=r.label,
            depth_cm=col_to_depth_cm(col, full_width, px_um),
            y_mm=r.centroid[0] * px_um / 1000,
            area_px=int(r.area),
            area_mm2=r.area * px_um ** 2 / 1e6,
            esd_um=r.equivalent_diameter_area * px_um,
            feret_max_um=fmax * px_um,
            feret_min_um=fmin * px_um,
            major_axis_um=r.axis_major_length * px_um,
            minor_axis_um=r.axis_minor_length * px_um,
            aspect_ratio=fmax / max(fmin, 1e-6),
            circularity=min(1.0, 4 * math.pi * r.area / max(perim, 1e-6) ** 2),
            solidity=r.solidity,
            orientation_deg=np.degrees(r.orientation),
            touches_edge=touches,
            edge_weight=edge_w,
        )
        rows.append(row)
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    if extra_rois:
        idx = df["chip_id"].values
        for name, roi in extra_rois.items():
            df[f"mean_{name}"] = ndi.mean(roi, lab, idx)
    if drop_edge:
        df = df[~df["touches_edge"] & np.isfinite(df["edge_weight"])].copy()
    else:
        df["edge_weight"] = 1.0
    return df.reset_index(drop=True)


# ---------- statistics -----------------------------------------------
def phi(d_um):
    return -np.log2(np.asarray(d_um, float) / 1000.0)

def size_values(df):
    v = df[SIZE_METRIC].to_numpy(float)
    return v * (4 / math.pi) if APPLY_STEREO else v

def chip_weights(df, sizes, weighting=None):
    weighting = weighting or WEIGHTING
    w = sizes ** 3 if weighting == "volume" else np.ones_like(sizes)
    return w * df["edge_weight"].to_numpy(float)

def wpercentile(values, weights, q):
    o = np.argsort(values)
    v, w = values[o], weights[o]
    cw = (np.cumsum(w) - 0.5 * w) / w.sum()
    return np.interp(np.asarray(q) / 100.0, cw, v)

def wentworth(d_um):
    for lim, name in [(4, "clay"), (63, "silt"), (125, "very fine sand"), (250, "fine sand"),
                      (500, "medium sand"), (1000, "coarse sand"), (2000, "very coarse sand"),
                      (4000, "granule")]:
        if d_um < lim:
            return name
    return "pebble"

def size_stats(df, weighting=None):
    if len(df) < 5:
        return dict(n_chips=len(df))
    s = size_values(df)
    w = chip_weights(df, s, weighting)
    D = dict(zip([5, 10, 16, 25, 50, 75, 84, 90, 95],
                 wpercentile(s, w, [5, 10, 16, 25, 50, 75, 84, 90, 95])))
    # Folk & Ward (1957): φx read from the coarse-to-fine cumulative curve -> φx = φ(D(100-x))
    f = {x: phi(D[100 - x]) for x in (5, 16, 25, 50, 75, 84, 95)}
    mz = (f[16] + f[50] + f[84]) / 3
    sorting = (f[84] - f[16]) / 4 + (f[95] - f[5]) / 6.6
    skew = ((f[16] + f[84] - 2 * f[50]) / (2 * (f[84] - f[16]))
            + (f[5] + f[95] - 2 * f[50]) / (2 * (f[95] - f[5])))
    kurt = (f[95] - f[5]) / (2.44 * (f[75] - f[25]))
    return dict(n_chips=len(df), D10_um=D[10], D50_um=D[50], D90_um=D[90],
                D50_class=wentworth(D[50]),
                FW_mean_um=1000 * 2 ** (-mz), FW_mean_phi=mz, FW_sorting_phi=sorting,
                FW_skewness=skew, FW_kurtosis=kurt,
                max_um=s.max(), mean_aspect=df["aspect_ratio"].mean(),
                mean_circularity=df["circularity"].mean())

print("Helpers ready.")


### Cell 3b — Which element map should you use?
Ranks every TIFF in the folder by how cleanly its brightness splits into two groups (chips vs matrix) and plots the best ones. Judge with your eyes too: pick the map where chip outlines look like chips. Mud rip-ups are usually clay-rich, so K, Ti, Al, Fe and Rb often work; Ca is usually the opposite (bright matrix, dark chips - then set `INVERT = True`). Set `BROWSE_XFRAC` to the part of the core that has the chips.

In [ ]:
# =====================================================================
# CELL 3b — Browse the element maps and see which one shows chips best
# =====================================================================
# Ranks every TIFF in the folder by how cleanly it splits into two
# populations (chips vs matrix, Otsu separability η) AND how much of that
# bright signal forms real objects rather than pixel noise (structure).
# chip score = η × structure. Still judge with your eyes: the best map is
# the one where the bright patches look like chips.
BROWSE_DIR   = "."            # folder holding the element maps
BROWSE_XFRAC = (0.0, 1.0)     # part of the core to judge, as a fraction of length
                              # e.g. (0.75, 1.0) = last quarter, where your chips are
BROWSE_TOP_N = 8              # how many maps to plot

browse_files = sorted(os.path.join(BROWSE_DIR, f) for f in os.listdir(BROWSE_DIR)
                      if f.lower().endswith((".tif", ".tiff")))
rows, thumbs = [], {}
for f in browse_files:
    try:
        i, a, _ = load_map(f)
    except Exception as e:
        print(f"  skipped {f}: {str(e)[:70]}"); continue
    Wf = i.shape[1]
    x0, x1 = int(BROWSE_XFRAC[0] * Wf), int(BROWSE_XFRAC[1] * Wf)
    i = i[:, x0:x1]
    try:
        _, v = core_mask(i)
    except Exception:
        continue
    if v.sum() < 1000:
        print(f"  skipped {f}: no core found"); continue
    sm = filters.gaussian(i, 0.7, preserve_range=True)
    s = 0.02 * i.shape[1]
    w_ = filters.gaussian(v.astype(float), s)
    m_ = filters.gaussian(np.where(v, sm, 0), s) / np.maximum(w_, 1e-6)
    flat = np.where(v, sm / np.maximum(m_, 1e-6), np.nan)
    vals = flat[v]
    thr = filters.threshold_otsu(vals)
    hi, lo = vals[vals > thr], vals[vals <= thr]
    if len(hi) < 50 or len(lo) < 50:
        continue
    p = len(hi) / len(vals)
    eta = p * (1 - p) * (hi.mean() - lo.mean()) ** 2 / vals.var()   # Otsu separability
    # structure = how much contrast survives blurring at chip scale.
    # Pixel noise disappears (low), real chips stay (high).
    filled = np.where(v, flat, vals.mean())
    structure = filters.gaussian(filled, 2.5, preserve_range=True)[v].var() / vals.var()
    rows.append(dict(file=f, element=os.path.splitext(os.path.basename(f))[0].split("_")[-1],
                     chip_score=eta * structure, separability=eta, structure=structure,
                     bright_fraction=100 * p, contrast=hi.mean() / lo.mean(),
                     px=f"{i.shape[1]}×{i.shape[0]}"))
    thumbs[f] = (flat, v)

rank = pd.DataFrame(rows).sort_values("chip_score", ascending=False).reset_index(drop=True)
display(rank.round(3))

top = rank.head(BROWSE_TOP_N)
h, w = thumbs[top.file.iloc[0]][0].shape
fig, axes = plt.subplots(len(top), 1, figsize=(14, max(1.0, 13 * h / w) * len(top)))
for ax, (_, r) in zip(np.atleast_1d(axes), top.iterrows()):
    flat, v = thumbs[r.file]
    ax.imshow(np.where(v, flat, np.nan), cmap="gray",
              vmin=np.nanpercentile(flat, 1), vmax=np.nanpercentile(flat, 99.5))
    ax.set_title(f"{r.element}   chip score = {r.chip_score:.3f}   "
                 f"(separability {r.separability:.2f} × structure {r.structure:.2f})   "
                 f"bright area = {r.bright_fraction:.0f} %", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/00_element_browse.png", dpi=150, bbox_inches="tight")
plt.show()
print("Put the file you like best into SEG_FILE in Cell 2, then re-run Cell 2 and continue.")


### Cell 4 — Load, calibrate, preview
Lists the pages inside the TIFF (set `SEG_PAGE` if the element you want is not page 0), then prints the pixel size and the smallest chip you can trust (3 px). The top panel is the map; the bottom is the signal used for segmentation with the core outline. Orange bands = your `INTERVALS`. Use the depth ruler to set them.

In [ ]:
# =====================================================================
# CELL 4 — Load map, calibrate pixel size, preview with a depth ruler
# =====================================================================
print("Pages in", SEG_FILE); list_pages(SEG_FILE)
inten, annot, rgb = load_map(SEG_FILE, invert=INVERT, page=SEG_PAGE)
H, W = inten.shape
print(f"{SEG_FILE}: {W} × {H} px")

# ---- pixel size -------------------------------------------------------
if PIXEL_SIZE_UM is not None:
    px_um = float(PIXEL_SIZE_UM)
    print(f"Pixel size from config: {px_um:.2f} µm/px")
else:
    if SCALEBAR_FILE and SCALEBAR_FILE != SEG_FILE:
        sb_int, sb_annot, _ = load_map(SCALEBAR_FILE)
        if sb_int.shape != inten.shape:
            raise ValueError(f"{SCALEBAR_FILE} is {sb_int.shape[::-1]} px but {SEG_FILE} is "
                             f"{inten.shape[::-1]} px - they must match to share a calibration.")
    else:
        sb_annot = annot
    sb = scalebar_length_px(sb_annot)
    if sb is None:
        raise ValueError(
            "No scale bar found in this image. Either set PIXEL_SIZE_UM (the mosaic step size "
            "from the M4 software - the most accurate option), or point SCALEBAR_FILE at the "
            "image that does carry the bar (e.g. 'Video Mosaic.tif'), same pixel size.")
    px_um = SCALEBAR_UM / sb["length_px"]
    print(f"Scale bar: {sb['length_px']} px = {SCALEBAR_UM/1e4:g} cm  ->  {px_um:.2f} µm/px")
    print("  (±1 px on the bar = ±%.1f%% on every size; the Tornado step size is more exact)"
          % (100 / sb["length_px"]))

RES_LIMIT_UM = MIN_DIAM_PX * px_um
print(f"Smallest chip that can be measured: ~{RES_LIMIT_UM:.0f} µm ({MIN_DIAM_PX} px)")

# ---- depth registration ------------------------------------------------
scan_len_px_cm = (W - 1) * px_um / 1e4
SCAN_LO, SCAN_HI = scan_depth_range(W, px_um)
print(f"\nScan covers core depth {SCAN_LO:.2f} - {SCAN_HI:.2f} cm "
      f"({'shallow end on the left' if CORE_TOP_SIDE == 'left' else 'shallow end on the right'})")
if SCAN_BOTTOM_DEPTH_CM is not None:
    span = abs(SCAN_BOTTOM_DEPTH_CM - SCAN_TOP_DEPTH_CM)
    diff = 100 * (scan_len_px_cm - span) / span
    print(f"  from your core log: {span:.2f} cm | from the pixel size: {scan_len_px_cm:.2f} cm "
          f"({diff:+.1f} %)")
    if abs(diff) > 5:
        print("  WARNING: these disagree by more than 5 %. Check the end depths and the pixel "
              "size - one of them is wrong, and depths of events will be shifted.")
else:
    print("  (length from pixel size; set SCAN_BOTTOM_DEPTH_CM as well to pin both ends)")
if SCAN_TOP_DEPTH_CM == 0.0 and SCAN_BOTTOM_DEPTH_CM is None:
    print("\n  *** SCAN_TOP_DEPTH_CM is still the placeholder 0 cm. Every depth below is "
          "\n      measured from the end of the scan, NOT from the core top. Set the real "
          "\n      depths in Cell 2, or run Cell 4b to find them from your XRF file. ***")

core, valid = core_mask(inten)
extra_full = {}
for name, path in EXTRA_MAPS.items():
    arr, _, _ = load_map(path)
    if arr.shape != inten.shape:
        raise ValueError(f"{path} is {arr.shape}, expected {inten.shape}")
    extra_full[name] = arr

# ---- preview ----------------------------------------------------------
depth_lo = col_to_depth_cm(0, W, px_um)
depth_hi = col_to_depth_cm(W - 1, W, px_um)
fig, ax = plt.subplots(2, 1, figsize=(16, 4.6), sharex=True,
                       gridspec_kw=dict(height_ratios=[1, 1]))
ax[0].imshow(rgb.astype(np.uint8) if rgb is not None else inten, cmap="gray",
             aspect="equal", extent=[0, W, H, 0])
ax[0].set_title(f"{CORE_ID} — {os.path.basename(SEG_FILE)}  ({px_um:.1f} µm/px)")
show = np.where(valid, inten, np.nan)
ax[1].imshow(show, cmap="gray", aspect="equal", extent=[0, W, H, 0])
ax[1].contour(core, [0.5], colors="cyan", linewidths=0.6)
ax[1].set_title("Signal used for segmentation (cyan = core mask; blank = voids / cracks)")

for a in ax:
    for name, top, bot in INTERVALS:
        c0, c1 = sorted([depth_to_col(top, W, px_um), depth_to_col(bot, W, px_um)])
        a.axvspan(c0, c1, color="orange", alpha=0.18)
        a.text((c0 + c1) / 2, 0, name, ha="center", va="bottom", color="darkorange", fontsize=8)
    a.set_yticks([])

step = 1.0
ticks_cm = np.arange(np.ceil(min(depth_lo, depth_hi)), max(depth_lo, depth_hi) + 1e-9, step)
ax[1].set_xticks([depth_to_col(d, W, px_um) for d in ticks_cm])
ax[1].set_xticklabels([f"{d:g}" for d in ticks_cm])
ax[1].set_xlabel("Core depth (cm)")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/01_preview_{CORE_ID}.png", dpi=200, bbox_inches="tight")
plt.show()


### Cell 4b — Where does the scan sit in the core? (optional)
Only needed if you do not know the depth range of the scan. Point `XRF_FILE` at your Avaatech spreadsheet and `MATCH_FILE` at the Tornado map of the same element (Ca is usually the strongest signal). It blurs the Tornado profile to the Avaatech spacing, slides it along the core, and reports the depth where the two curves agree best, in both orientations. **Look at the plot before believing the number** - a good match is one sharp correlation peak and two curves whose wiggles line up. Then paste the three lines it prints into Cell 2 and re-run Cells 2-4.

In [ ]:
# =====================================================================
# CELL 4b — Where does this scan sit in the core? (optional)
# =====================================================================
# Slides the Tornado element profile along your Avaatech XRF depth series
# and finds the position where the two agree best. Skipped if XRF_FILE is None.
if XRF_FILE:
    # --- 1. the Avaatech series -----------------------------------------
    if XRF_FILE.lower().endswith((".xlsx", ".xls", ".xlsm")):
        xrf = pd.read_excel(XRF_FILE, sheet_name=XRF_SHEET)
    else:
        xrf = pd.read_csv(XRF_FILE, sep=None, engine="python")
    xrf.columns = [str(c).strip() for c in xrf.columns]
    dcol = XRF_DEPTH_COL or next((c for c in xrf.columns if "depth" in c.lower()), None)
    if dcol is None:
        raise ValueError(f"No depth column found. Columns: {list(xrf.columns)}")
    ecol = next((c for c in xrf.columns if c.lower() == XRF_ELEMENT.lower()), None) \
        or next((c for c in xrf.columns if XRF_ELEMENT.lower() in c.lower()), None)
    if ecol is None:
        raise ValueError(f"No column for '{XRF_ELEMENT}'. Columns: {list(xrf.columns)}")
    xd = pd.to_numeric(xrf[dcol], errors="coerce") * (0.1 if XRF_UNITS.lower() == "mm" else 1.0)
    xv = pd.to_numeric(xrf[ecol], errors="coerce")
    ok = xd.notna() & xv.notna()
    xd, xv = xd[ok].to_numpy(), xv[ok].to_numpy()
    o = np.argsort(xd); xd, xv = xd[o], xv[o]
    dz = np.median(np.diff(xd))
    print(f"XRF: {len(xd)} points, {xd.min():.2f}-{xd.max():.2f} cm, step {dz:.2f} cm, "
          f"matching on '{ecol}'")

    # --- 2. the Tornado profile along the scan --------------------------
    m_int, _, _ = load_map(MATCH_FILE, page=SEG_PAGE) if MATCH_FILE else (inten, None, None)
    if m_int.shape != inten.shape:
        raise ValueError(f"{MATCH_FILE} is a different size from {SEG_FILE}")
    cnt = valid.sum(axis=0)                               # mean per column, ignoring voids
    prof = np.where(cnt > 0, np.where(valid, m_int, 0).sum(axis=0) / np.maximum(cnt, 1), np.nan)
    prof = pd.Series(prof).interpolate(limit_direction="both").to_numpy()
    prof = ndi.gaussian_filter1d(prof, max(1.0, (dz / 2) * 1e4 / px_um))   # blur to XRF resolution
    scan_len = (W - 1) * px_um / 1e4

    # --- 3. slide it along and score ------------------------------------
    def score(offset, flip):
        p = prof[::-1] if flip else prof
        pos = offset + np.linspace(0, scan_len, W)        # depth of each column
        sel = (xd >= pos[0]) & (xd <= pos[-1])
        if sel.sum() < max(8, 0.5 * scan_len / dz):
            return np.nan
        a = np.interp(xd[sel], pos, p)
        b = xv[sel]
        if a.std() == 0 or b.std() == 0:
            return np.nan
        return float(np.corrcoef(a, b)[0, 1])

    offsets = np.arange(xd.min() - 0.25 * scan_len, xd.max() - 0.5 * scan_len, 0.05)
    res = [(off, flip, score(off, flip)) for flip in (False, True) for off in offsets]
    res = [r for r in res if np.isfinite(r[2])]
    if not res:
        raise ValueError("No overlap - check XRF_UNITS and the depth column.")
    best_off, best_flip, best_r = max(res, key=lambda r: abs(r[2]))
    top_side = ("right" if CORE_TOP_SIDE == "left" else "left") if best_flip else CORE_TOP_SIDE
    print(f"\nBest match: r = {best_r:+.2f}  ->  scan covers {best_off:.2f} - "
          f"{best_off + scan_len:.2f} cm, shallow end on the {top_side}")
    print("Paste into Cell 2 (only if the curves below really line up):")
    print(f'    CORE_TOP_SIDE        = "{top_side}"')
    print(f"    SCAN_TOP_DEPTH_CM    = {best_off:.2f}")
    print(f"    SCAN_BOTTOM_DEPTH_CM = {best_off + scan_len:.2f}")
    if abs(best_r) < 0.5:
        print("  WARNING: weak match. Try another element, or set the depths by hand.")
    if best_r < 0:
        print("  NOTE: the match is negative (one curve is the mirror of the other). That happens "
              "when the Tornado colour scale is inverted - check the plot before trusting it.")

    # --- 4. plot ---------------------------------------------------------
    p = prof[::-1] if best_flip else prof
    pos = best_off + np.linspace(0, scan_len, W)
    fig, ax = plt.subplots(figsize=(12, 3.2))
    ax.plot(xd, (xv - xv.mean()) / xv.std(), color="#4C72B0", lw=1.4, label=f"Avaatech {ecol}")
    ax.plot(pos, (p - p.mean()) / p.std(), color="#DD8452", lw=1.2,
            label=f"Tornado {os.path.basename(MATCH_FILE or SEG_FILE)}")
    ax.set_xlabel("Core depth (cm)"); ax.set_ylabel("normalised signal")
    ax.set_title(f"Depth registration: best r = {best_r:+.2f} at {best_off:.2f} cm")
    ax.legend(fontsize=8)
    ax.set_xlim(best_off - 2, best_off + scan_len + 2)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/01b_depth_match_{CORE_ID}.png", dpi=200, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(figsize=(12, 2.2))
    for flip, c, lab in [(False, "#4C72B0", "as set"), (True, "#DD8452", "flipped")]:
        ax.plot(offsets, [score(o_, flip) for o_ in offsets], color=c, lw=1, label=lab)
    ax.axvline(best_off, color="k", ls="--", lw=1)
    ax.set_xlabel("scan top depth (cm)"); ax.set_ylabel("correlation")
    ax.set_title("How sharp is the match? A single clear peak = trustworthy")
    ax.legend(fontsize=8); plt.tight_layout(); plt.show()
else:
    print("XRF_FILE is None - using the depths you typed in Cell 2 "
          f"({SCAN_TOP_DEPTH_CM} cm to "
          f"{SCAN_BOTTOM_DEPTH_CM if SCAN_BOTTOM_DEPTH_CM is not None else 'pixel-size end'}).")


### Cell 4c — Events
Reads your event file (CSV or XLSX) and turns every event into a measuring interval, so all results come out per event instead of per hand-typed depth range. It finds the name / top / bottom / type columns by name, flips top and bottom if they are the wrong way round, converts mm to cm if you say so, and clips events to the scanned window - telling you which events are only partly scanned. `EVENT_PAD_CM` adds a margin above and below each event if you want to catch chips just outside the picked boundary.

In [ ]:
# =====================================================================
# CELL 4b — Read the event file and turn each event into an interval
# =====================================================================
def _pick(cols, *cands):
    low = {c.lower().strip(): c for c in cols}
    for cand in cands:
        for k, orig in low.items():
            if k == cand or k.startswith(cand) or cand in k.replace("_", " "):
                return orig
    return None

if EVENT_FILE:
    if EVENT_FILE.lower().endswith((".xlsx", ".xls", ".xlsm")):
        ev = pd.read_excel(EVENT_FILE, sheet_name=EVENT_SHEET)
    else:
        ev = pd.read_csv(EVENT_FILE, sep=None, engine="python")
    ev.columns = [str(c).strip() for c in ev.columns]
    c_name = _pick(ev.columns, "event", "name", "id", "label", "unit")
    c_top  = _pick(ev.columns, "top", "upper", "start", "from", "depth top")
    c_bot  = _pick(ev.columns, "bottom", "base", "lower", "end", "to", "depth bottom")
    c_type = _pick(ev.columns, "type", "class", "trigger", "interpretation")
    if c_top is None or c_bot is None:
        raise ValueError(f"Could not find top/bottom depth columns in {EVENT_FILE}.\n"
                         f"Columns found: {list(ev.columns)}\n"
                         "Rename them to something like 'Event, Top_cm, Bottom_cm, Type'.")
    k = 0.1 if EVENT_UNITS.lower() == "mm" else 1.0
    events = pd.DataFrame({
        "event": (ev[c_name].astype(str) if c_name else
                  [f"E{i+1}" for i in range(len(ev))]),
        "top_cm": pd.to_numeric(ev[c_top], errors="coerce") * k,
        "bottom_cm": pd.to_numeric(ev[c_bot], errors="coerce") * k,
        "type": ev[c_type].astype(str) if c_type else "",
    }).dropna(subset=["top_cm", "bottom_cm"])
    # make sure top is the shallower number
    flip = events.top_cm > events.bottom_cm
    events.loc[flip, ["top_cm", "bottom_cm"]] = events.loc[flip, ["bottom_cm", "top_cm"]].values
    events["thickness_cm"] = events.bottom_cm - events.top_cm

    # which events fall inside the scanned window?
    events["measure_top_cm"] = (events.top_cm - EVENT_PAD_CM).clip(lower=SCAN_LO)
    events["measure_bottom_cm"] = (events.bottom_cm + EVENT_PAD_CM).clip(upper=SCAN_HI)
    events["in_scan"] = events.measure_bottom_cm - events.measure_top_cm > 0.2
    events["partly_outside"] = events.in_scan & (
        (events.top_cm - EVENT_PAD_CM < SCAN_LO - 1e-9) | (events.bottom_cm + EVENT_PAD_CM > SCAN_HI + 1e-9))

    display(events.round(2))
    inside = events[events.in_scan]
    print(f"{len(inside)} of {len(events)} events fall inside the scanned window "
          f"({SCAN_LO:.2f}-{SCAN_HI:.2f} cm)")
    if inside.partly_outside.any():
        print("  partly outside the scan (stats cover only the scanned part): "
              + ", ".join(inside[inside.partly_outside].event))
    if inside.empty:
        raise ValueError("No event overlaps the scanned depth window. Check SCAN_TOP_DEPTH_CM / "
                         "SCAN_BOTTOM_DEPTH_CM in Cell 2, or the units of the event file.")
    INTERVALS = [(r.event, r.measure_top_cm, r.measure_bottom_cm) for _, r in inside.iterrows()]
else:
    events = pd.DataFrame(columns=["event", "top_cm", "bottom_cm", "type", "thickness_cm"])
    print("No event file - using the INTERVALS list from Cell 2.")

print("\nWill measure:", ", ".join(f"{n} ({a:.2f}-{b:.2f} cm)" for n, a, b in INTERVALS))


### Cell 5 — Segment and measure + QC
**Always look at this figure before trusting the numbers.** Red outlines = measured chips, blue = excluded because they touch an edge. The zoom panel colours each chip by size. If red outlines sit on matrix, raise `CONTRAST_MIN` or use `THRESH_METHOD = "li"`/`"manual"`; if obvious chips are missing, lower `CONTRAST_MIN`; if single chips are cut in two, raise `SPLIT_H_PX`.

In [ ]:
# =====================================================================
# CELL 5 — Find and measure mud chips in each interval + QC overlay
# =====================================================================
def run_interval(name, top, bot, params=None, keep_images=False):
    p = default_params() if params is None else params
    c0, c1 = sorted([depth_to_col(top, W, px_um), depth_to_col(bot, W, px_um)])
    roi, roi_valid, roi_core = inten[:, c0:c1], valid[:, c0:c1], core[:, c0:c1]
    lab, flat, thr = segment_roi(roi, roi_valid, px_um, p)
    extra = {k: v[:, c0:c1] for k, v in extra_full.items()}
    df = measure_chips(lab, roi_core, c0, W, px_um, extra, drop_edge=p["drop_edge"])
    if not df.empty:
        df.insert(0, "interval", name)
        df.insert(1, "core_id", CORE_ID)
    kept = np.isin(lab, df["chip_id"].values) if not df.empty else np.zeros_like(lab, bool)
    info = dict(interval=name, top_cm=top, bottom_cm=bot, col0=c0, col1=c1, threshold=thr,
                valid_area_mm2=roi_valid.sum() * px_um ** 2 / 1e6,
                chip_area_pct=100 * (lab > 0).sum() / max(roi_valid.sum(), 1))
    if keep_images:
        info.update(roi=roi, lab=lab, kept=kept)
    return df, info


all_chips, infos = [], []
for name, top, bot in INTERVALS:
    df_i, info = run_interval(name, top, bot, keep_images=True)
    all_chips.append(df_i); infos.append(info)
    print(f"{name}: {len(df_i)} chips measured, threshold = {info['threshold']:.3f}, "
          f"chip area = {info['chip_area_pct']:.1f}% of sediment")
chips = pd.concat(all_chips, ignore_index=True)


# ---- QC figure: every chip outlined, coloured by size -----------------
def qc_figure(info, df):
    roi, lab, kept = info["roi"], info["lab"], info["kept"]
    size_map = np.full(lab.shape, np.nan)
    if not df.empty:
        lut = np.full(lab.max() + 1, np.nan)
        lut[df["chip_id"].values] = df[SIZE_METRIC].values
        size_map = np.where(kept, lut[lab], np.nan)
    v = np.clip(roi / np.percentile(roi[roi > 0], 99.5), 0, 1)
    bnd_kept = segmentation.find_boundaries(np.where(kept, lab, 0), mode="inner")
    bnd_drop = segmentation.find_boundaries(np.where(~kept, lab, 0), mode="inner")
    over = np.dstack([v] * 3)
    over[bnd_kept] = [1, 0.1, 0.1]            # red   = measured
    over[bnd_drop] = [0.2, 0.6, 1.0]          # blue  = rejected (edge-cut)

    h, w = lab.shape
    side = min(h, w // 2)                                  # square zoom window in the middle
    zc0, zr0 = (w - side) // 2, (h - side) // 2
    zs = (slice(zr0, zr0 + side), slice(zc0, zc0 + side))
    strip_h = 14 * h / w
    fig = plt.figure(figsize=(14, 2 * strip_h + 6))
    gs = fig.add_gridspec(3, 1, height_ratios=[strip_h, strip_h, 6])
    ax = [fig.add_subplot(gs[i]) for i in range(3)]
    ax[0].imshow(v, cmap="gray"); ax[0].set_title(f"{info['interval']} — raw signal")
    ax[1].imshow(over)
    ax[1].set_title("Red = measured chip outline · Blue = touches core/interval edge (excluded)")
    ax[1].add_patch(plt.Rectangle((zc0, zr0), side, side, fill=False, ec="yellow", lw=1))
    ax[2].imshow(over[zs], interpolation="nearest")
    im = ax[2].imshow(size_map[zs], cmap="viridis", alpha=0.55, interpolation="nearest")
    ax[2].set_title(f"Zoom (yellow box) — colour = {SIZE_METRIC}")
    plt.colorbar(im, ax=ax[2], fraction=0.03, pad=0.01, label="µm")
    for a in ax:
        a.set_yticks([]); a.set_xticks([])
    plt.tight_layout()
    safe = "".join(c if c.isalnum() or c in "-_" else "_" for c in str(info["interval"]))
    plt.savefig(f"{OUT_DIR}/02_QC_{CORE_ID}_{safe}.png", dpi=200, bbox_inches="tight")
    plt.show()

for info in infos:
    qc_figure(info, chips[chips["interval"] == info["interval"]])


### Cell 6 — Statistics and distributions
Summary table per interval. Left plot: volume-% (bars) and number-% (line) in ¼-φ classes; red dotted line = resolution limit (anything left of it is under-sampled). Right plot: cumulative curve with D10/D50/D90.

*Comparing with Malvern:* ESD with volume weighting is the closest match to laser-diffraction volume %. Remember these are **2-D section** sizes — a random cut through a chip is on average smaller than the chip itself; `APPLY_STEREO = True` applies the 4/π sphere correction if you want an estimate of true size.

In [ ]:
# =====================================================================
# CELL 6 — Grain-size statistics + distribution plots per interval
# =====================================================================
rows = []
for info in infos:
    d = chips[chips["interval"] == info["interval"]]
    st = size_stats(d)
    rows.append(dict(core_id=CORE_ID, interval=info["interval"],
                     top_cm=info["top_cm"], bottom_cm=info["bottom_cm"],
                     pixel_um=round(px_um, 2), res_limit_um=round(RES_LIMIT_UM),
                     size_metric=SIZE_METRIC, weighting=WEIGHTING, stereo_corrected=APPLY_STEREO,
                     chip_area_pct=info["chip_area_pct"],
                     chips_per_cm2=len(d) / (info["valid_area_mm2"] / 100), **st))
summary = pd.DataFrame(rows).rename(columns={"interval": "event"})
if len(events):                       # bring the event metadata alongside the stats
    meta = events[["event", "top_cm", "bottom_cm", "thickness_cm", "type", "partly_outside"]]
    summary = summary.drop(columns=["top_cm", "bottom_cm"]).merge(meta, on="event", how="left")
    front = ["core_id", "event", "type", "top_cm", "bottom_cm", "thickness_cm", "partly_outside"]
    summary = summary[front + [c for c in summary.columns if c not in front]]
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
display(summary.T if len(summary) <= 3 else summary.round(2))

# ---- plots ------------------------------------------------------------
n = len(infos)
fig, axes = plt.subplots(n, 2, figsize=(12, 3.6 * n), squeeze=False)
for k, info in enumerate(infos):
    d = chips[chips["interval"] == info["interval"]]
    if len(d) < 5:
        print(f"{info['interval']}: only {len(d)} chips - too few for a distribution"); continue
    s = size_values(d); w = chip_weights(d, s)
    ph = phi(s)
    bins_phi = np.arange(np.floor(ph.min() * 4) / 4, np.ceil(ph.max() * 4) / 4 + 0.26, 0.25)  # ¼-φ classes
    st = summary[summary.event == info["interval"]].iloc[0]

    ax = axes[k, 0]
    h_, e_ = np.histogram(ph, bins=bins_phi, weights=w)
    ax.bar(e_[:-1], 100 * h_ / h_.sum(), width=0.25, align="edge",
           color="#4C72B0", edgecolor="white")
    hn, _ = np.histogram(ph, bins=bins_phi)
    ax.step(e_[:-1], 100 * hn / hn.sum(), where="post", color="k", lw=1, label="number %")
    ax.set_xlim(bins_phi[0], bins_phi[-1])      # coarse (low φ) on the left
    ax.set_xlabel("φ  (coarser ←   → finer)")
    ax.set_ylabel(f"{WEIGHTING} %")
    ax.set_title(f"{info['interval']}  (n = {len(d)})")
    ax.axvline(phi(RES_LIMIT_UM), color="red", ls=":", lw=1)
    ax.text(phi(RES_LIMIT_UM), ax.get_ylim()[1] * 0.95, " resolution\n limit",
            color="red", fontsize=7, va="top")
    ax.legend(fontsize=7, loc="upper left")

    ax = axes[k, 1]
    o = np.argsort(s)
    cum = 100 * np.cumsum(w[o]) / w.sum()
    ax.semilogx(s[o], cum, color="#4C72B0", lw=2)
    for q, c in [("D10_um", "#999"), ("D50_um", "k"), ("D90_um", "#999")]:
        ax.axvline(st[q], color=c, ls="--", lw=1)
        ax.text(st[q], 3, f" {q[:3]}\n {st[q]:.0f} µm", fontsize=7)
    for lim in (63, 125, 250, 500, 1000, 2000, 4000):
        ax.axvline(lim, color="#ddd", lw=0.6, zorder=0)
    ax.set_xlabel(f"{SIZE_METRIC.replace('_um','')} (µm)")
    ax.set_ylabel(f"cumulative {WEIGHTING} % finer")
    ax.set_title(f"D50 = {st['D50_um']:.0f} µm ({st['D50_class']}), "
                 f"sorting σ = {st['FW_sorting_phi']:.2f} φ")
    ax.set_ylim(0, 100)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/03_distributions_{CORE_ID}.png", dpi=200, bbox_inches="tight")
plt.show()


### Cell 7 — Down-core profile
Depth runs down the page, core top at the top, with every event shaded. Panels: each chip as a dot, then `BIN_MM` (2 mm) bins of D50/D90, chip area % and chips per cm². This is where you see whether chips concentrate at the event base and fine upward - the pattern that separates event types.

In [ ]:
# =====================================================================
# CELL 7 — Down-core profile in BIN_MM bins, with the events marked
# =====================================================================
prof_rows = []
for info in infos:
    d = chips[chips["interval"] == info["interval"]]
    edges = np.arange(info["top_cm"], info["bottom_cm"] + 1e-9, BIN_MM / 10)
    if len(edges) < 2 or edges[-1] < info["bottom_cm"] - 1e-9:
        edges = np.append(edges, info["bottom_cm"])
    for a, b in zip(edges[:-1], edges[1:]):
        db = d[(d["depth_cm"] >= a) & (d["depth_cm"] < b)]
        c0, c1 = sorted([depth_to_col(a, W, px_um), depth_to_col(b, W, px_um)])
        sed_mm2 = valid[:, c0:c1].sum() * px_um ** 2 / 1e6
        row = dict(core_id=CORE_ID, event=info["interval"], top_cm=a, bottom_cm=b,
                   mid_cm=(a + b) / 2, n_chips=len(db),
                   chip_area_pct=100 * db["area_mm2"].sum() / max(sed_mm2, 1e-9),
                   chips_per_cm2=len(db) / max(sed_mm2 / 100, 1e-9))
        if len(db) >= 5:
            st = size_stats(db)
            row.update(D50_um=st["D50_um"], D90_um=st["D90_um"], max_um=st["max_um"])
        elif len(db) > 0:
            row.update(max_um=size_values(db).max())
        prof_rows.append(row)
profile = pd.DataFrame(prof_rows)
for col in ("D50_um", "D90_um", "max_um"):
    if col not in profile:
        profile[col] = np.nan

# ---- figure: depth vertical, core top at the top ----------------------
ylo = min(i["top_cm"] for i in infos) - 0.5
yhi = max(i["bottom_cm"] for i in infos) + 0.5
fig, ax = plt.subplots(1, 4, figsize=(13, max(6, 0.55 * (yhi - ylo))), sharey=True)

for a in ax:                                   # shade every event band
    for _, r in (events[events.in_scan] if len(events) else pd.DataFrame()).iterrows():
        a.axhspan(r.top_cm, r.bottom_cm, color="#F0C089", alpha=0.35, zorder=0)
        a.axhline(r.bottom_cm, color="#B07020", lw=0.6, zorder=1)

ax[0].scatter(chips[SIZE_METRIC], chips["depth_cm"], s=5, alpha=0.35, color="#4C72B0")
ax[0].axvline(RES_LIMIT_UM, color="red", ls=":", lw=1)
ax[0].set_xscale("log"); ax[0].set_xlabel(f"every chip\n{SIZE_METRIC} (µm)")
ax[0].set_ylabel("Core depth (cm)")

for ev_name, g in profile.groupby("event"):
    g = g.sort_values("mid_cm")
    ax[1].plot(g["D50_um"], g["mid_cm"], "-o", color="k", ms=3)
    ax[1].plot(g["D90_um"], g["mid_cm"], "--o", color="gray", ms=3)
    ax[2].plot(g["chip_area_pct"], g["mid_cm"], "-o", color="#DD8452", ms=3)
    ax[3].plot(g["chips_per_cm2"], g["mid_cm"], "-o", color="#55A868", ms=3)
ax[1].plot([], [], "-o", color="k", ms=3, label="D50")
ax[1].plot([], [], "--o", color="gray", ms=3, label="D90")
ax[1].legend(fontsize=7)
ax[1].set_xlabel(f"binned size (µm)\n{WEIGHTING}-weighted")
ax[2].set_xlabel("chip area\n(% of sediment)")
ax[3].set_xlabel("chip density\n(chips per cm²)")

if len(events):
    for _, r in events[events.in_scan].iterrows():
        ax[3].text(ax[3].get_xlim()[1], (r.top_cm + r.bottom_cm) / 2, f" {r.event}",
                   va="center", fontsize=7, color="#805000")
ax[0].set_ylim(yhi, ylo)                        # core top at the top
for a in ax:
    a.grid(axis="y", lw=0.3, alpha=0.4)
fig.suptitle(f"{CORE_ID} — mud chips vs depth ({BIN_MM:g} mm bins; shaded = events)")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/04_profile_{CORE_ID}.png", dpi=200, bbox_inches="tight")
plt.show()
display(profile.round(2).head(20))


### Cell 7b — Per-event figure: Tornado map next to the depth/size plot
For every event this puts the µXRF Tornado picture — raw signal and the same red/blue
chip-outline QC overlay as Cell 5 — **rotated so depth runs top-to-bottom**, directly beside a
depth-vs-chip-size panel with that event's band shaded and D50/D90 traced through it. Same
depth axis on both sides, so you can look across from a bright patch on the scan straight to the
point it made on the size curve. One figure per event, saved to `OUT_DIR`.


In [ ]:
# =====================================================================
# CELL 7b — Per-event figure: Tornado map next to the depth/size profile
# =====================================================================
# Left: the same raw-signal + red/blue QC overlay as Cell 5, rotated so
# depth runs down the page. Right: every chip's size vs depth, D50/D90
# traced through BIN_MM bins, with this event's band shaded — both
# panels share the same depth axis so the picture and the numbers line up.
SHALLOW_AT_COL0 = col_to_depth_cm(0, W, px_um) <= col_to_depth_cm(W - 1, W, px_um)

def _rotate_to_depth(img):
    """Rotate a (cross-core, along-core) slice so depth runs top -> bottom."""
    rot = np.rot90(img, k=-1)                       # left edge of img -> top row of rot
    return rot if SHALLOW_AT_COL0 else np.flipud(rot)

def event_figure(info, df, profile):
    roi, lab, kept = info["roi"], info["lab"], info["kept"]
    top, bot = info["top_cm"], info["bottom_cm"]

    v = np.clip(roi / np.percentile(roi[roi > 0], 99.5), 0, 1)
    bnd_kept = segmentation.find_boundaries(np.where(kept, lab, 0), mode="inner")
    bnd_drop = segmentation.find_boundaries(np.where(~kept, lab, 0), mode="inner")
    over = np.dstack([v] * 3)
    over[bnd_kept] = [1, 0.1, 0.1]                   # red  = measured chip
    over[bnd_drop] = [0.2, 0.6, 1.0]                 # blue = excluded (edge-cut)

    img_v, over_v = _rotate_to_depth(v), _rotate_to_depth(over)
    w_v = img_v.shape[1]

    fig = plt.figure(figsize=(11, max(6, 0.55 * (bot - top))))
    gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 2.2], wspace=0.08)
    axL1 = fig.add_subplot(gs[0])
    axL2 = fig.add_subplot(gs[1], sharey=axL1)
    axR = fig.add_subplot(gs[2], sharey=axL1)

    axL1.imshow(img_v, cmap="gray", aspect="auto", extent=[0, w_v, bot, top])
    axL1.set_title("raw signal (Al)", fontsize=9)
    axL2.imshow(over_v, aspect="auto", extent=[0, w_v, bot, top])
    axL2.set_title("red = chip · blue = excluded", fontsize=8)
    for a in (axL1, axL2):
        a.set_xticks([])
    axL1.set_ylabel("Core depth (cm)")

    axR.scatter(df[SIZE_METRIC], df["depth_cm"], s=6, alpha=0.4, color="#4C72B0",
                label="every chip")
    axR.axvline(RES_LIMIT_UM, color="red", ls=":", lw=1, label="resolution limit")
    g = profile[profile.event == info["interval"]].sort_values("mid_cm")
    if len(g):
        axR.plot(g["D50_um"], g["mid_cm"], "-o", color="k", ms=3, label="D50")
        axR.plot(g["D90_um"], g["mid_cm"], "--o", color="gray", ms=3, label="D90")
    axR.axhspan(top, bot, color="#F0C089", alpha=0.25, zorder=0)
    axR.set_xscale("log")
    axR.set_xlabel(f"{SIZE_METRIC} (µm)")
    axR.legend(fontsize=7, loc="lower right")

    axL1.set_ylim(bot, top)                          # shallow end at the top, on every panel
    for a in (axL1, axL2, axR):
        a.grid(axis="y", lw=0.3, alpha=0.3)
    ev_type = ""
    if len(events):
        m = events[events.event == info["interval"]]
        if len(m) and str(m.iloc[0]["type"]).strip() not in ("", "nan"):
            ev_type = f" — {m.iloc[0]['type']}"
    fig.suptitle(f"{CORE_ID} — {info['interval']}{ev_type}  ({top:.2f}-{bot:.2f} cm)", y=1.02)
    plt.tight_layout()
    safe = "".join(c if c.isalnum() or c in "-_" else "_" for c in str(info["interval"]))
    plt.savefig(f"{OUT_DIR}/05_event_{CORE_ID}_{safe}.png", dpi=200, bbox_inches="tight")
    plt.show()

for info in infos:
    event_figure(info, chips[chips["interval"] == info["interval"]], profile)


### Cell 8 — Accuracy / sensitivity check
Re-runs the whole interval with the threshold ±5 % and the contrast rule ±0.10 (9 runs). The half-range of D50 is a defensible method uncertainty to report. For an independent check, trace ~30–50 chips by hand in ImageJ/Fiji on the same map and compare their ESDs with `mudchips_*_all_chips.csv`.

In [ ]:
# =====================================================================
# CELL 8 — Accuracy check: how much do D-values move if the settings move?
# =====================================================================
# Re-runs segmentation with the threshold ±5 % and the contrast rule ±0.10.
# The spread of D50 across these runs is an honest "method uncertainty"
# to quote next to your numbers (e.g. D50 = 850 ± 60 µm).
baseline_n = chips.groupby("interval").size()
sens_rows = []
for name, top, bot in INTERVALS:
    if baseline_n.get(name, 0) < 5:
        print(f"{name}: only {baseline_n.get(name, 0)} chips - skipped"); continue
    for ts in (0.95, 1.00, 1.05):
        for cm_ in (CONTRAST_MIN - 0.10, CONTRAST_MIN, CONTRAST_MIN + 0.10):
            p = default_params(); p.update(thresh_scale=ts, contrast=cm_)
            d_, info_ = run_interval(name, top, bot, params=p)
            st = size_stats(d_)
            sens_rows.append(dict(event=name, thresh_scale=ts, contrast_min=round(cm_, 2),
                                  n_chips=st.get("n_chips"), D10_um=st.get("D10_um"),
                                  D50_um=st.get("D50_um"), D90_um=st.get("D90_um"),
                                  chip_area_pct=info_["chip_area_pct"]))
sens = pd.DataFrame(sens_rows)
if sens.empty:
    print("Nothing to test - no event had enough chips.")
display(sens.round(2))

unc = (sens.groupby("event")[["n_chips", "D10_um", "D50_um", "D90_um", "chip_area_pct"]]
           .agg(["min", "max", "std"]))
display(unc)
for name, g in sens.groupby("event"):
    base = g[(g.thresh_scale == 1.0) & np.isclose(g.contrast_min, CONTRAST_MIN)].iloc[0]
    half = (g["D50_um"].max() - g["D50_um"].min()) / 2
    if not np.isfinite(base.D50_um) or not np.isfinite(half):
        print(f"{name}: too few chips for a stable D50"); continue
    print(f"{name}: D50 = {base.D50_um:.0f} ± {half:.0f} µm "
          f"({100*half/base.D50_um:.0f} % method sensitivity)"
          + ("   <-- large: check the QC figure, this event has few chips"
             if 100 * half / base.D50_um > 20 else ""))


### Cell 9 — Export
Writes per-chip, summary, profile and sensitivity CSVs plus all figures, zips them and downloads the zip.

In [ ]:
# =====================================================================
# CELL 9 — Save tables + figures and download everything as one zip
# =====================================================================
chips.to_csv(f"{OUT_DIR}/mudchips_{CORE_ID}_all_chips.csv", index=False)
summary.to_csv(f"{OUT_DIR}/mudchips_{CORE_ID}_per_event_summary.csv", index=False)
profile.to_csv(f"{OUT_DIR}/mudchips_{CORE_ID}_profile_{BIN_MM:g}mm.csv", index=False)
if len(events):
    events.to_csv(f"{OUT_DIR}/mudchips_{CORE_ID}_events_used.csv", index=False)
if "sens" in globals():
    sens.to_csv(f"{OUT_DIR}/mudchips_{CORE_ID}_sensitivity.csv", index=False)

zip_name = f"mudchips_{CORE_ID}.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(f"{OUT_DIR}/*")):
        z.write(f)
print("Saved:", *sorted(os.listdir(OUT_DIR)), sep="\n  ")
if IN_COLAB:
    files.download(zip_name)
